# Stage 01 - Define and display the study area

This notebook imports the Parcel A boundary, validates its geometry and area, displays it on a map, and saves a standardized version for every subsequent stage.

**Expected result:** one authoritative AOI stored in `EPSG:32633` for analysis and `EPSG:4326` for data exchange and Google Earth Engine.

> The default boundary is reconstructed from seven vertices in the French report. It remains a candidate until formal confirmation and therefore does not pass quality gate G0.

In [ ]:
# Install packages in Google Colab. This runs once per session.
%pip install -q geopandas fiona pyogrio shapely pyproj folium earthengine-api geemap

In [ ]:
from pathlib import Path
import json
import shutil

import pandas as pd
import geopandas as gpd
gpd.options.io_engine = 'fiona'
import folium
from folium import plugins
from IPython.display import display
from shapely.geometry import Polygon
from shapely.validation import explain_validity

#@title Input settings
SOURCE_MODE = 'report_vertices'  #@param ['report_vertices', 'upload', 'earth_engine_asset']
SOURCE_CRS_IF_MISSING = 'EPSG:32633'  #@param {type:'string'}
EE_ASSET_ID = ''  #@param {type:'string'}
GEE_PROJECT_ID = ''  #@param {type:'string'}
AOI_APPROVED = False  #@param {type:'boolean'}

ANALYSIS_CRS = 'EPSG:32633'
EXCHANGE_CRS = 'EPSG:4326'
REPORTED_AREA_HA = 5128.69

REPORT_VERTICES_UTM33N = [
    ('A1', 375142.89553203, 730553.78023993, 927),
    ('A2', 381315.67865558, 726934.57048810, 952),
    ('A3', 386655.47274849, 730036.88441500, 933),
    ('A4', 381124.21918453, 733184.44527635, 918),
    ('A5', 382254.47034824, 735241.34560231, 914),
    ('A6', 380256.00132039, 736073.12628858, 912),
    ('A7', 376031.75713711, 732703.05038135, 925),
]

print('Input mode:', SOURCE_MODE)

## Select the boundary source

- `report_vertices`: run immediately with the seven report vertices; this is the default mode.
- `upload`: upload a GeoJSON, GPKG, KML, or ZIP containing a Shapefile.
- `earth_engine_asset`: read a FeatureCollection from a personal or organizational Earth Engine Asset.

When an official boundary becomes available, use the second or third mode. No other code needs to change.

In [ ]:
def load_uploaded_vector():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('Run upload mode in Google Colab or provide a local file path directly in the code.') from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one vector file or one ZIP archive per run.')
    name, content = next(iter(uploaded.items()))
    path = Path('/content') / name
    path.write_bytes(content)
    if path.suffix.lower() == '.zip':
        return gpd.read_file(f'zip://{path}')
    if path.suffix.lower() == '.kml':
        return gpd.read_file(path, engine='pyogrio')
    return gpd.read_file(path)


if SOURCE_MODE == 'report_vertices':
    coordinates = [(row[1], row[2]) for row in REPORT_VERTICES_UTM33N]
    source_gdf = gpd.GeoDataFrame(
        [{'aoi_id': 'parcel_a', 'source': 'final_report_table_1'}],
        geometry=[Polygon(coordinates)],
        crs=ANALYSIS_CRS,
    )
    source_status = 'reconstructed_from_report'
elif SOURCE_MODE == 'upload':
    source_gdf = load_uploaded_vector()
    source_status = 'uploaded_candidate'
elif SOURCE_MODE == 'earth_engine_asset':
    if not EE_ASSET_ID or not GEE_PROJECT_ID:
        raise ValueError('EE_ASSET_ID and GEE_PROJECT_ID are required.')
    import ee
    import geemap
    try:
        ee.Initialize(project=GEE_PROJECT_ID)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT_ID)
    source_gdf = geemap.ee_to_gdf(ee.FeatureCollection(EE_ASSET_ID))
    source_status = 'earth_engine_candidate'
else:
    raise ValueError(f'Unknown SOURCE_MODE: {SOURCE_MODE}')

if source_gdf.crs is None:
    source_gdf = source_gdf.set_crs(SOURCE_CRS_IF_MISSING)

print('Input feature count:', len(source_gdf))
print('Input CRS:', source_gdf.crs)

In [ ]:
# Clean, dissolve, and validate the geometry
working = source_gdf.loc[~source_gdf.geometry.is_empty & source_gdf.geometry.notna()].copy()
if working.empty:
    raise ValueError('The input contains no usable geometry.')

working_utm = working.to_crs(ANALYSIS_CRS)
try:
    merged_geometry = working_utm.geometry.union_all()
except AttributeError:
    merged_geometry = working_utm.geometry.unary_union

if not merged_geometry.is_valid:
    from shapely import make_valid
    merged_geometry = make_valid(merged_geometry)

if merged_geometry.geom_type not in {'Polygon', 'MultiPolygon'}:
    raise ValueError(f'The final geometry must be a Polygon or MultiPolygon, not {merged_geometry.geom_type}.')

aoi_utm = gpd.GeoDataFrame(
    [{
        'aoi_id': 'parcel_a',
        'status': source_status,
        'approved': bool(AOI_APPROVED),
        'source_mode': SOURCE_MODE,
    }],
    geometry=[merged_geometry],
    crs=ANALYSIS_CRS,
)
aoi_wgs84 = aoi_utm.to_crs(EXCHANGE_CRS)

area_ha = float(aoi_utm.geometry.area.iloc[0] / 10_000)
perimeter_km = float(aoi_utm.geometry.length.iloc[0] / 1_000)
difference_ha = area_ha - REPORTED_AREA_HA
difference_pct = difference_ha / REPORTED_AREA_HA * 100
is_valid = bool(aoi_utm.geometry.is_valid.iloc[0])
validity_message = explain_validity(aoi_utm.geometry.iloc[0])
geometry_checks_pass = is_valid and area_ha > 0
g0_gate = 'PASS' if geometry_checks_pass and AOI_APPROVED else 'HOLD'

centroid_utm = aoi_utm.geometry.centroid.iloc[0]
centroid_wgs84 = gpd.GeoSeries([centroid_utm], crs=ANALYSIS_CRS).to_crs(EXCHANGE_CRS).iloc[0]

validation = pd.DataFrame([
    {'check': 'Valid geometry', 'value': is_valid, 'result': 'PASS' if is_valid else 'FAIL'},
    {'check': 'Geometry type', 'value': merged_geometry.geom_type, 'result': 'PASS'},
    {'check': 'Calculated area (ha)', 'value': round(area_ha, 4), 'result': 'PASS'},
    {'check': 'Reported area (ha)', 'value': REPORTED_AREA_HA, 'result': 'REFERENCE'},
    {'check': 'Area difference (%)', 'value': round(difference_pct, 5), 'result': 'REVIEW'},
    {'check': 'Formal boundary approval', 'value': AOI_APPROVED, 'result': 'PASS' if AOI_APPROVED else 'HOLD'},
    {'check': 'G0 quality gate', 'value': g0_gate, 'result': g0_gate},
])
display(validation)
print('Geometry validity:', validity_message)

In [ ]:
# Display the AOI interactively
center = [centroid_wgs84.y, centroid_wgs84.x]
aoi_map = folium.Map(location=center, zoom_start=12, tiles='OpenStreetMap', control_scale=True)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery',
    name='Esri satellite imagery',
    overlay=False,
).add_to(aoi_map)

boundary_layer = folium.GeoJson(
    data=json.loads(aoi_wgs84.to_json()),
    name='AOI boundary',
    style_function=lambda _: {
        'color': '#24573a', 'weight': 4, 'fillColor': '#80b918', 'fillOpacity': 0.30
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['aoi_id', 'status', 'approved'],
        aliases=['AOI', 'Source status', 'Approved'],
    ),
).add_to(aoi_map)

if SOURCE_MODE == 'report_vertices':
    vertex_frame = gpd.GeoDataFrame(
        [{'vertex_id': row[0], 'altitude_m': row[3]} for row in REPORT_VERTICES_UTM33N],
        geometry=gpd.points_from_xy(
            [row[1] for row in REPORT_VERTICES_UTM33N],
            [row[2] for row in REPORT_VERTICES_UTM33N],
        ),
        crs=ANALYSIS_CRS,
    ).to_crs(EXCHANGE_CRS)
    for row in vertex_frame.itertuples():
        folium.CircleMarker(
            [row.geometry.y, row.geometry.x], radius=5, color='#9d2a2a',
            fill=True, fill_color='white', fill_opacity=1,
            tooltip=f'{row.vertex_id} — {row.altitude_m} m',
        ).add_to(aoi_map)

plugins.Fullscreen(position='topleft').add_to(aoi_map)
plugins.MeasureControl(primary_length_unit='kilometers', primary_area_unit='hectares').add_to(aoi_map)
folium.LayerControl(collapsed=False).add_to(aoi_map)
minx, miny, maxx, maxy = aoi_wgs84.total_bounds
aoi_map.fit_bounds([[miny, minx], [maxy, maxx]])
display(aoi_map)

In [ ]:
# Save standardized outputs
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'config/project.yaml').exists() else Path('/content/parcel-a-agri-geospatial')
DATA_DIR = PROJECT_ROOT / 'data/aoi'
MAP_DIR = PROJECT_ROOT / 'outputs/maps'
TABLE_DIR = PROJECT_ROOT / 'outputs/tables'
for directory in (DATA_DIR, MAP_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

version_label = 'v1' if g0_gate == 'PASS' else 'candidate'
geojson_path = DATA_DIR / f'aoi_{version_label}_wgs84.geojson'
gpkg_path = DATA_DIR / f'aoi_{version_label}_utm33n.gpkg'
summary_path = DATA_DIR / f'aoi_{version_label}_summary.json'
map_path = MAP_DIR / f'01_aoi_{version_label}.html'
validation_path = TABLE_DIR / '01_aoi_validation.csv'

aoi_wgs84.to_file(geojson_path, driver='GeoJSON')
aoi_utm.to_file(gpkg_path, layer='aoi', driver='GPKG')
validation.to_csv(validation_path, index=False, encoding='utf-8-sig')
aoi_map.save(map_path)

summary = {
    'aoi_id': 'parcel_a',
    'source_mode': SOURCE_MODE,
    'source_status': source_status,
    'analysis_crs': ANALYSIS_CRS,
    'exchange_crs': EXCHANGE_CRS,
    'geometry_type': merged_geometry.geom_type,
    'is_valid': is_valid,
    'area_ha': area_ha,
    'reported_area_ha': REPORTED_AREA_HA,
    'area_difference_ha': difference_ha,
    'area_difference_percent': difference_pct,
    'perimeter_km': perimeter_km,
    'centroid_wgs84': [centroid_wgs84.x, centroid_wgs84.y],
    'approved': bool(AOI_APPROVED),
    'g0_gate': g0_gate,
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('Outputs saved under:', PROJECT_ROOT)
for path in (geojson_path, gpkg_path, summary_path, map_path, validation_path):
    print(' -', path)

In [ ]:
#@title Optional display of this AOI in Google Earth Engine
CONNECT_TO_EE = False  #@param {type:'boolean'}

if CONNECT_TO_EE:
    if not GEE_PROJECT_ID:
        raise ValueError('Enter GEE_PROJECT_ID in the settings cell.')
    import ee
    import geemap
    try:
        ee.Initialize(project=GEE_PROJECT_ID)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT_ID)
    ee_aoi = geemap.geopandas_to_ee(aoi_wgs84)
    ee_map = geemap.Map(center=center, zoom=12)
    ee_map.add_basemap('SATELLITE')
    ee_map.addLayer(ee_aoi.style(color='00A651', fillColor='80B91855', width=3), {}, 'Parcel A AOI')
    ee_map.centerObject(ee_aoi, 12)
    display(ee_map)
else:
    print('Earth Engine is optional in Stage 01 and was not initialized.')

## Interpret the result and decide whether to proceed

- If the map matches the official boundary, set `AOI_APPROVED=True` only after confirmation by the responsible project representative. The output will be stored as `aoi_v1`, and G0 will change to PASS.
- If an official Shapefile, GeoJSON, KML, or Earth Engine Asset exists, import it and review differences in shape and area.
- While G0 remains HOLD, code and the data inventory may be prepared, but definitive agricultural statistics must not be published.

**Next stage after approval:** build the comprehensive data inventory and test coverage of GAEZ v5, WaPOR, SoilFER, Crop Suitability App, CAVA, and prioritized non-FAO sources for this AOI.